## MR Complexity Analysis (Unsimplified Trees): Tree Metrics + Halstead + McCabe

This notebook parses each Metamorphic Relation (MR) written in text form (`LHS => RHS`) into an **unsimplified parse tree** (i.e., it preserves the original expressions as leaf nodes instead of abstracting them). It then computes **structural tree metrics**, **McCabe cyclomatic complexity (adapted)**, and **Halstead metrics** over the tree, enabling comparisons across **domains** and across **sources** (**Human** vs **AI**). Finally, it exports per-domain and global results as CSV, plus human-readable reports and summary tables (PDF/LaTeX).

### Workflow
1. **Load MRs per domain** from `mrs_<domain>.txt` files (one MR per line).
2. **Parse each MR** into a tree with root `MR` and children `LHS` / `RHS`.
   - Logical connectors are represented explicitly as `AND` and `OR` nodes.
   - Atomic conditions become:
     - `COMP(op)` for comparisons (`>`, `>=`, `<=`, `<`, `==`, `!=`)
     - `PRED` / `NEG_PRED` for predicates (and negated predicates)
     - `EXPR` as a fallback for expressions that do not match the above
   - Leaf nodes keep the **full original expressions** (e.g., `NS(m1)`, `TTD(m2)`, `D(m1)`).
3. **Compute metrics** for each MR (tree metrics, McCabe, Halstead).
4. **Export outputs**: per-domain CSVs + combined CSV, text reports with trees, and summary tables in PDF/LaTeX.

---

## Metrics (definitions consistent with the notebook)

### Tree structural metrics
- **Tree_Size**: total number of nodes in the tree (`tree.size()`), including `MR`, `LHS`, `RHS`, operators, and leaves.
- **Depth**: maximum depth in **edges** (`tree.depth()`), where a leaf has depth `0`.  
  (So depth equals the length of the longest root-to-leaf path in edges.)
- **Leaves**: number of leaf nodes (`tree.leaves()`), typically the atomic expressions.
- **Branching_Factor**: average branching factor computed as:

    Branching_Factor = (Tree_Size - 1) / (#internal_nodes)
    #internal_nodes = Tree_Size - Leaves

  (In the notebook: edges / internal nodes.)

### McCabe (Cyclomatic Complexity, adapted)
- **McCabe_CC**: computed as:

    McCabe_CC = (# of OR nodes) + 1

  (Only `OR` increases the number of alternative logical paths.)

### Halstead (computed over the tree)
Halstead is computed over tree labels with:
- **Operators**: `AND`, `OR`, `COMP(...)`, `PRED`, `NEG_PRED`, `EXPR`
- **Operands**: all other labels (i.e., the original expressions)
- **Structural nodes** (`MR`, `LHS`, `RHS`) are excluded.

Let:
- `N1` = total operators, `n1` = distinct operators
- `N2` = total operands,  `n2` = distinct operands
- `N = N1 + N2` (length), `n = n1 + n2` (vocabulary)

Then:
- **Halstead_V (Volume)**:

    V = N * log2(n)

- **Halstead_D (Difficulty)**:

    D = (n1 / 2) * (N2 / n2)

- **Halstead_E (Effort)**:

    E = D * V

(The notebook rounds `V`, `D`, and `E` to 2 decimals.)

---

## Worked example (tree + metric computations)

### example-human-MR1
- **Rule**: `NS(m1) > NS(m2)  =>  TTD(m2) >= TTD(m1) AND D(m2) >= D(m1)`
- **Tree**:
```text
MR
  LHS
    COMP(>)
      NS(m1)
      NS(m2)
  RHS
    AND
      COMP(>=)
        TTD(m2)
        TTD(m1)
      COMP(>=)
        D(m2)
        D(m1)
```
### Metric computations

#### 1) Tree_Size
Count all nodes:
1. MR
2. LHS
3. COMP(>)
4. NS(m1)
5. NS(m2)
6. RHS
7. AND
8. COMP(>=) [1]
9. TTD(m2)
10. TTD(m1)
11. COMP(>=) [2]
12. D(m2)
13. D(m1)

    Tree_Size = 13

#### 2) Depth (edges; leaf depth = 0)
Longest root-to-leaf path:
MR → RHS → AND → COMP(>=) → D(m1)

This path has 4 edges, therefore:

    Depth = 4

#### 3) Leaves
Leaf expressions:
NS(m1), NS(m2), TTD(m2), TTD(m1), D(m2), D(m1)

    Leaves = 6

#### 4) Branching_Factor
Using the notebook formula:

    #internal_nodes = Tree_Size - Leaves = 13 - 6 = 7
    edges = Tree_Size - 1 = 12
    Branching_Factor = edges / #internal_nodes = 12 / 7 ≈ 1.714 → 1.71

    Branching_Factor ≈ 1.71

#### 5) McCabe_CC
Number of OR nodes = 0

    McCabe_CC = 0 + 1 = 1

#### 6) Halstead (V, D, E)
**Operators**: COMP(>), AND, COMP(>=), COMP(>=)
- N1 = 4
- n1 = 3  (distinct: COMP(>), AND, COMP(>=))

**Operands**: NS(m1), NS(m2), TTD(m2), TTD(m1), D(m2), D(m1)
- N2 = 6
- n2 = 6

Derived:
- N = N1 + N2 = 4 + 6 = 10
- n = n1 + n2 = 3 + 6 = 9

Volume:
    V = N * log2(n) = 10 * log2(9)
      = 10 * 3.169925... ≈ 31.6993 → 31.70

Difficulty:
    D = (n1/2) * (N2/n2) = (3/2) * (6/6) = 1.5 → 1.50

Effort:
    E = D * V = 1.5 * 31.6993 ≈ 47.5489 → 47.55

    Halstead_V ≈ 31.70
    Halstead_D = 1.50
    Halstead_E ≈ 47.55

In [167]:
"""
MR Complexity Analysis (Unsimplified Trees)
=============================================
Computes complexity metrics on MR parse trees WITHOUT abstracting
operands to skeleton labels. The full original expressions are
preserved as tree nodes.

Metrics:
- Halstead Volume, Difficulty, Effort (Halstead, 1977)
- McCabe Cyclomatic Complexity (McCabe, 1976)
- Tree metrics: Size, Depth, Leaves, Branching Factor

References:
- Halstead, M.H. (1977). Elements of Software Science. North Holland.
- McCabe, T.J. (1976). A complexity measure. IEEE Trans. SE-2(4), 308-320.
"""

import numpy as np
import pandas as pd
from dataclasses import dataclass, field
from typing import List
import matplotlib.pyplot as plt
import re
import os
import math
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

In [168]:
# ============================================================
# CONFIGURATION

In [169]:
# ============================================================
INPUT_FILES = [
    "mrs_DCs_human_extra.txt",
    "mrs_AVs_human_extra.txt",
    "mrs_DFAs_human_extra.txt",
    "mrs_Words.txt",
]
OUTPUT_DIR = "output_complexity"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [170]:
# ============================================================
# TREE NODE (unsimplified)

In [171]:
# ============================================================
@dataclass
class TreeNode:
    label: str
    children: List['TreeNode'] = field(default_factory=list)

    def size(self):
        return 1 + sum(c.size() for c in self.children)

    def depth(self):
        if not self.children:
            return 0
        return 1 + max(c.depth() for c in self.children)

    def leaves(self):
        if not self.children:
            return 1
        return sum(c.leaves() for c in self.children)

    def to_string(self, indent=0):
        s = '  ' * indent + self.label + '\n'
        for c in self.children:
            s += c.to_string(indent + 1)
        return s

    def print_tree(self, prefix='', is_last=True):
        connector = '└── ' if is_last else '├── '
        print(prefix + connector + self.label)
        prefix += '    ' if is_last else '│   '
        for i, child in enumerate(self.children):
            child.print_tree(prefix, i == len(self.children) - 1)

In [172]:
# ============================================================
# PARSER: MR text -> unsimplified tree

In [173]:
# ============================================================
# The tree structure is:
#   MR
#   ├── LHS
#   │   ├── AND (if multiple conditions)
#   │   │   ├── <comparison_node>  e.g. COMP(>, NS(m1), NS(m2))
#   │   │   └── <comparison_node>
#   │   └── (or single comparison)
#   └── RHS
#       └── (same structure, with OR support)
#
# comparison_node for numeric: COMP(op, left_expr, right_expr)
# comparison_node for boolean: PRED(R(x1,w1)) or NEG_PRED(¬R(x1,w1))
# comparison_node for method:  METHOD_COMP(op, expr1, expr2) e.g. size()<size()

def tokenize_expression(text):
    """Extract meaningful tokens from an expression, preserving function calls."""
    text = text.strip()
    tokens = []
    # Find all function calls like NS(m1), TTD(m2), FS(x1)->size(), etc.
    # and operators, literals
    i = 0
    while i < len(text):
        if text[i].isspace():
            i += 1
            continue
        # Check for comparison operators
        if text[i:i+2] in ('>=', '<=', '==', '!='):
            tokens.append(text[i:i+2])
            i += 2
        elif text[i] in ('>', '<', '='):
            tokens.append(text[i])
            i += 1
        else:
            # Grab everything until next operator or space
            j = i
            paren_depth = 0
            while j < len(text):
                if text[j] == '(':
                    paren_depth += 1
                elif text[j] == ')':
                    paren_depth -= 1
                    if paren_depth < 0:
                        break
                elif paren_depth == 0 and (text[j] in ' \t' or text[j:j+2] in ('>=', '<=', '==', '!=') or text[j] in ('>', '<', '=')):
                    break
                j += 1
            token = text[i:j].strip()
            if token:
                tokens.append(token)
            i = j
    return tokens


def parse_comparison(text):
    """Parse a single comparison expression into a tree node.
    
    Examples:
      NS(m1) > NS(m2)           -> COMP(>) with children [NS(m1), NS(m2)]
      W(m1)=W(m2)               -> COMP(=) with children [W(m1), W(m2)]
      FS(x1)->size() < FS(x2)->size() -> COMP(<) with children [FS(x1)->size(), FS(x2)->size()]
      R(x1, w1)                 -> PRED with child [R(x1,w1)]
      ¬R(x1, w1)                -> NEG_PRED with child [R(x1,w1)]
      w1->last() == '0'         -> COMP(==) with children [w1->last(), '0']
      w1->including('1') == w2  -> COMP(==) with children [w1->including('1'), w2]
      NFS(x1) == FS(x2)         -> COMP(==) with children [NFS(x1), FS(x2)]
    """
    text = text.strip()

    # Handle negated predicates: ¬R(x1, w1), ¬Result(x1, w1)
    if text.startswith('¬'):
        inner = text[1:].strip()
        return TreeNode('NEG_PRED', [TreeNode(inner)])

    # Check for comparison operator
    # Need to find operator not inside parentheses
    ops = ['>=', '<=', '==', '!=', '>', '<', '=']
    best_pos = -1
    best_op = None
    paren_depth = 0
    i = 0
    while i < len(text):
        if text[i] == '(':
            paren_depth += 1
        elif text[i] == ')':
            paren_depth -= 1
        elif paren_depth == 0:
            for op in ops:
                if text[i:i+len(op)] == op:
                    # Skip -> (method call arrow, not comparison)
                    if op in ('>', '>=') and i > 0 and text[i-1] == '-':
                        break
                    # Skip if this is part of =>
                    if op == '>' and i + 1 < len(text) and text[i+1] == '=':
                        break  # will be caught by >=
                    if op == '<' and i + 1 < len(text) and text[i+1] == '=':
                        break  # will be caught by <=
                    if op == '=' and i > 0 and text[i-1] in ('>', '<', '!', '='):
                        break  # part of >=, <=, !=, ==
                    if op == '=' and i + 1 < len(text) and text[i+1] == '=':
                        break  # part of ==
                    if op == '=' and i > 0 and text[i-1] == '>':
                        break  # part of =>
                    best_pos = i
                    best_op = op
                    break
            if best_pos >= 0:
                break
        i += 1

    if best_pos >= 0:
        left = text[:best_pos].strip()
        right = text[best_pos + len(best_op):].strip()
        op_label = best_op if best_op != '=' else '=='
        return TreeNode(f'COMP({op_label})', [TreeNode(left), TreeNode(right)])

    # No operator found - it's a predicate
    # R(x1, w1), Result(x1, w1), etc.
    if re.match(r'\w+\(', text):
        return TreeNode('PRED', [TreeNode(text)])

    # Fallback
    return TreeNode('EXPR', [TreeNode(text)])


def split_top_level(text, separator):
    """Split text by separator, respecting parentheses."""
    result = []
    last = 0
    paren_depth = 0
    # Use case-insensitive search for 'and'/'or'
    sep_lower = separator.lower()
    sep_len = len(separator)
    i = 0
    while i < len(text):
        if text[i] == '(':
            paren_depth += 1
        elif text[i] == ')':
            paren_depth -= 1
        elif paren_depth == 0:
            chunk = text[i:i+sep_len]
            if chunk.lower() == sep_lower:
                # Check word boundaries
                before_ok = (i == 0) or not text[i-1].isalnum()
                after_ok = (i + sep_len >= len(text)) or not text[i + sep_len].isalnum()
                if before_ok and after_ok:
                    result.append(text[last:i].strip())
                    last = i + sep_len
                    i = last
                    continue
        i += 1
    result.append(text[last:].strip())
    return [r for r in result if r]


def strip_outer_parens(text):
    """Remove matching outer parentheses."""
    text = text.strip()
    if text.startswith('(') and text.endswith(')'):
        depth = 0
        for i, c in enumerate(text):
            if c == '(':
                depth += 1
            elif c == ')':
                depth -= 1
            if depth == 0 and i < len(text) - 1:
                return text  # parens don't match fully
        return text[1:-1].strip()
    return text


def parse_side(text, side_label):
    """Parse one side (LHS or RHS) of an MR into a tree.
    
    Supports AND, OR connectors and nested expressions.
    """
    text = text.strip()

    # Try OR first (lower precedence)
    or_parts = split_top_level(text, 'or')
    if len(or_parts) > 1:
        children = []
        for part in or_parts:
            part = strip_outer_parens(part)
            # Each OR branch might have ANDs
            children.append(parse_side_inner(part))
        return TreeNode(side_label, [TreeNode('OR', children)])

    # Then AND
    return TreeNode(side_label, [parse_side_inner(text)])


def parse_side_inner(text):
    """Parse inner expression (after OR split) handling AND."""
    text = text.strip()
    text = strip_outer_parens(text)

    and_parts = split_top_level(text, 'and')
    if len(and_parts) > 1:
        children = [parse_comparison(strip_outer_parens(p)) for p in and_parts]
        return TreeNode('AND', children)

    return parse_comparison(text)


def parse_mr_to_tree(raw_text):
    """Parse a full MR line into an unsimplified tree.
    
    Format: group, MRn, category, LHS_expr '=>  RHS_expr
    """
    clean = raw_text.replace("'=>", "=>").replace("' =>", "=>")
    if '=>' not in clean:
        raise ValueError(f'No => found in: {raw_text}')

    parts = clean.split('=>', 1)
    left_part = parts[0].strip()
    rhs_text = parts[1].strip()

    # Extract LHS text (skip group, MR id, category)
    comma_parts = left_part.split(',')
    if len(comma_parts) >= 4:
        lhs_text = ','.join(comma_parts[3:]).strip()
    elif len(comma_parts) >= 3:
        lhs_text = ','.join(comma_parts[2:]).strip()
    else:
        lhs_text = left_part

    lhs_node = parse_side(lhs_text, 'LHS')
    rhs_node = parse_side(rhs_text, 'RHS')

    return TreeNode('MR', [lhs_node, rhs_node])


def parse_mr_line(line):
    """Extract MR id, raw text, and origin (Human/AI) from a line."""
    line = line.strip()
    if not line or line.startswith('#'):
        return None
    clean = line.replace("'=>", "=>").replace("' =>", "=>")
    if '=>' not in clean:
        return None
    left = clean.split('=>', 1)[0].strip()
    cp = left.split(',')

    # Detect 'human' in any field position
    fields = [f.strip().lower() for f in cp]
    is_human = 'human' in fields

    if len(cp) >= 3:
        group = cp[0].strip()
        p1, p2 = cp[1].strip(), cp[2].strip()
        if p1.startswith('MR'):
            mr_id, category = p1, p2
        else:
            category, mr_id = p1, p2
    else:
        group, mr_id, category = 'default', 'MR', 'normal'

    origin = 'Human' if is_human else 'AI'
    return f'{group}-{category}-{mr_id}', line, origin


def load_mrs_from_file(filepath):
    """Load and parse all MRs from a file."""
    with open(filepath, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    trees = {}
    raw_texts = {}
    origins = {}
    errors = []
    for line in lines:
        result = parse_mr_line(line)
        if result is None:
            continue
        full_id, raw_text, origin = result
        if full_id in trees:
            continue
        try:
            trees[full_id] = parse_mr_to_tree(raw_text)
            raw_texts[full_id] = raw_text
            origins[full_id] = origin
        except Exception as e:
            errors.append((full_id, raw_text, str(e)))
    return trees, raw_texts, origins, errors

In [174]:
# ============================================================
# COMPLEXITY METRICS (on unsimplified trees)

In [175]:
# ============================================================

# Operators: logical connectors AND, OR, comparison operators COMP(>), COMP(<=), etc.
# Operands: everything else that is a leaf or expression node
# Structural: MR, LHS, RHS (excluded from Halstead)

STRUCTURAL_NODES = {'MR', 'LHS', 'RHS'}

def collect_all_nodes(node):
    """Collect all nodes from a tree."""
    result = [node]
    for c in node.children:
        result.extend(collect_all_nodes(c))
    return result


def is_operator(label):
    """An operator is AND, OR, or any COMP(...), PRED, NEG_PRED, EXPR."""
    if label in ('AND', 'OR'):
        return True
    if label.startswith('COMP('):
        return True
    if label in ('PRED', 'NEG_PRED', 'EXPR'):
        return True
    return False


def is_structural(label):
    return label in STRUCTURAL_NODES


def is_operand(label):
    """An operand is any leaf-level token that is not structural or operator."""
    return not is_operator(label) and not is_structural(label)


def compute_halstead(node):
    """
    Halstead metrics on the unsimplified tree.
    
    Operators: AND, OR, COMP(>), COMP(<=), COMP(==), PRED, NEG_PRED
    Operands: actual expressions - NS(m1), TTD(m2), R(x1,w1), '0', etc.
    Structural (MR, LHS, RHS): excluded.
    
    Reference: Halstead, M.H. (1977). Elements of Software Science.
    """
    all_nodes = collect_all_nodes(node)
    all_labels = [n.label for n in all_nodes]

    operators = [l for l in all_labels if is_operator(l)]
    operands = [l for l in all_labels if is_operand(l)]

    N1 = len(operators)
    N2 = len(operands)
    n1 = len(set(operators))
    n2 = len(set(operands))
    N = N1 + N2
    n = n1 + n2

    V = N * math.log2(n) if n > 1 else 0.0
    D = (n1 / 2) * (N2 / n2) if n2 > 0 else 0.0
    E = D * V

    return {
        'N1_operators': N1,
        'N2_operands': N2,
        'n1_unique_ops': n1,
        'n2_unique_opds': n2,
        'N_length': N,
        'n_vocabulary': n,
        'Halstead_V': round(V, 2),
        'Halstead_D': round(D, 2),
        'Halstead_E': round(E, 2),
    }


def compute_mccabe(node):
    """
    McCabe CC on unsimplified tree.
    CC = number of OR nodes + 1
    
    Reference: McCabe, T.J. (1976). A complexity measure.
    """
    all_labels = [n.label for n in collect_all_nodes(node)]
    return sum(1 for l in all_labels if l == 'OR') + 1


def compute_branching_factor(node):
    """Average branching factor = edges / internal nodes."""
    total = node.size()
    leaf_count = node.leaves()
    internal = total - leaf_count
    if internal == 0:
        return 0.0
    return (total - 1) / internal


def compute_all_metrics(mr_id, tree, raw_text):
    """Compute all metrics for one MR."""
    halstead = compute_halstead(tree)
    cc = compute_mccabe(tree)

    # Collect operator and operand details for the report
    all_nodes = collect_all_nodes(tree)
    ops = [n.label for n in all_nodes if is_operator(n.label)]
    opds = [n.label for n in all_nodes if is_operand(n.label)]

    return {
        'MR_ID': mr_id,
        'Raw_MR': raw_text.strip(),
        'Tree_Size': tree.size(),
        'Depth': tree.depth(),
        'Leaves': tree.leaves(),
        'Branching_Factor': round(compute_branching_factor(tree), 2),
        'McCabe_CC': cc,
        **halstead,
        'Operators_detail': ', '.join(ops),
        'Operands_detail': ', '.join(opds),
    }

In [176]:
# ============================================================
# TABLE PDF GENERATION

In [177]:
# ============================================================
def generate_table_pdf(df, output_path):
    """Generate a single PDF with one metrics table per domain."""
    domains = df['Domain'].unique()
    n_domains = len(domains)
    fig, axes = plt.subplots(n_domains, 1,
                              figsize=(18, 2.5 + 1.1 * len(df)),
                              gridspec_kw={'hspace': 0.45})
    if n_domains == 1:
        axes = [axes]

    display_cols = ['MR_ID', 'Tree_Size', 'Depth', 'Leaves',
                    'n_vocabulary', 'Halstead_V', 'McCabe_CC']
    col_labels = ['MR ID', 'Size', 'Depth', 'Leaves',
                  'Vocabulary', 'Halstead V', 'McCabe CC']

    for idx, (domain, ax) in enumerate(zip(domains, axes)):
        ax.axis('off')
        sub = df[df['Domain'] == domain][display_cols].copy()
        sub = sub.sort_values('MR_ID')

        # Summary row
        num_cols = ['Tree_Size', 'Depth', 'Leaves', 'n_vocabulary',
                    'Halstead_V', 'McCabe_CC']
        summary_row = {'MR_ID': '── MEAN ──'}
        for col in num_cols:
            val = sub[col].astype(float).mean()
            summary_row[col] = f'{val:.2f}'
        summary = pd.DataFrame([summary_row])
        sub = pd.concat([sub, summary], ignore_index=True)

        cell_text = sub.values.tolist()
        table = ax.table(cellText=cell_text, colLabels=col_labels,
                         cellLoc='center', loc='center')
        table.auto_set_font_size(False)
        table.set_fontsize(7.5)
        table.scale(1.0, 1.35)

        # Style header
        for j in range(len(col_labels)):
            table[0, j].set_facecolor('#2171B5')
            table[0, j].set_text_props(color='white', fontweight='bold')

        # Style summary row
        last_row = len(cell_text)
        for j in range(len(col_labels)):
            table[last_row, j].set_facecolor('#E8E8E8')
            table[last_row, j].set_text_props(fontweight='bold')

        col_widths = [0.20, 0.08, 0.08, 0.08, 0.10, 0.12, 0.10]
        for j, w in enumerate(col_widths):
            for i in range(len(cell_text) + 1):
                table[i, j].set_width(w)

        ax.set_title(f'{domain}', fontsize=11, fontweight='bold', pad=8,
                     loc='left', color='#333333')

    fig.suptitle('MR Complexity Metrics (Unsimplified Trees)\n'
                 'Halstead (1977) · McCabe (1976)',
                 fontsize=13, fontweight='bold', y=0.98)
    plt.savefig(output_path, format='pdf', dpi=300, bbox_inches='tight')
    plt.close()
    print(f'  Saved: {output_path}')

In [178]:
# ============================================================
# REPORT GENERATION

In [179]:
# ============================================================
def generate_report(domain, trees, raw_texts, metrics_df, output_path):
    """Generate a text report with tree visualizations and metrics."""
    lines = []
    lines.append('=' * 70)
    lines.append(f'MR COMPLEXITY REPORT (UNSIMPLIFIED TREES): {domain}')
    lines.append('=' * 70)
    lines.append('')
    lines.append('References:')
    lines.append('  Halstead, M.H. (1977). Elements of Software Science. North Holland.')
    lines.append('  McCabe, T.J. (1976). A complexity measure. IEEE Trans. SE-2(4), 308-320.')
    lines.append('')
    lines.append('Halstead classification for MR trees:')
    lines.append('  Operators: AND, OR, COMP(>), COMP(>=), COMP(<=), COMP(<),')
    lines.append('             COMP(==), COMP(!=), PRED, NEG_PRED')
    lines.append('  Operands:  Original expressions (NS(m1), TTD(m2), R(x1,w1), etc.)')
    lines.append('  Structural (MR, LHS, RHS): excluded from Halstead.')
    lines.append('')

    lines.append('-' * 50)
    lines.append('SUMMARY')
    lines.append('-' * 50)
    lines.append(f'Total MRs: {len(trees)}')
    for col in ['Tree_Size', 'Depth', 'Halstead_V', 'McCabe_CC']:
        vals = metrics_df[col].astype(float)
        lines.append(f'  {col}: mean={vals.mean():.2f}, std={vals.std():.2f}, '
                      f'min={vals.min():.2f}, max={vals.max():.2f}')

    lines.append('')
    lines.append('-' * 50)
    lines.append('PER-MR DETAILS')
    lines.append('-' * 50)

    for mr_id, tree in trees.items():
        lines.append(f'\n{mr_id}:')
        raw = raw_texts.get(mr_id, '')
        # Extract just the rule part
        if '=>' in raw:
            rule_part = raw.split(',', 2)[-1].strip() if ',' in raw else raw
            # Further clean: take from after last category comma
            clean = raw.replace("'=>", "=>").replace("' =>", "=>")
            parts = clean.split(',')
            if len(parts) >= 4:
                rule_part = ','.join(parts[3:]).strip().replace("=>", " => ")
            elif len(parts) >= 3:
                rule_part = ','.join(parts[2:]).strip().replace("=>", " => ")
            lines.append(f'  Rule: {rule_part}')

        lines.append(f'  Tree:')
        for tl in tree.to_string().strip().split('\n'):
            lines.append(f'    {tl}')

        row = metrics_df[metrics_df['MR_ID'] == mr_id].iloc[0]
        lines.append(f'  Size={int(row["Tree_Size"])}, Depth={int(row["Depth"])}, '
                      f'Leaves={int(row["Leaves"])}, '
                      f'Vocabulary={int(row["n_vocabulary"])}')
        lines.append(f'  Halstead V={row["Halstead_V"]:.2f}, '
                      f'D={row["Halstead_D"]:.2f}, E={row["Halstead_E"]:.2f}')
        lines.append(f'  McCabe CC={int(row["McCabe_CC"])}')
        lines.append(f'  Operators: {row["Operators_detail"]}')
        lines.append(f'  Operands:  {row["Operands_detail"]}')

    lines.append('\n' + '=' * 70)
    lines.append('END OF REPORT')
    lines.append('=' * 70)

    with open(output_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(lines))
    print(f'  Saved: {output_path}')

In [180]:
# ============================================================
# LATEX TABLE GENERATION

In [181]:
def generate_latex_table(df, output_path):
    """
    Generate LaTeX table: complexity summary per domain, Human vs AI.
    New format: siunitx S columns with avg/min/max per metric (TS, Depth, V, D, E).
    """

    # Keep domain order as they appear in the dataframe
    domains = df["Domain"].dropna().unique().tolist()

    # Metric columns in df, ordered as in the LaTeX table
    metric_map = [
        ("Tree_Size", ("TS", "S[table-format=2.2]", "S[table-format=2.2]", "S[table-format=2.2]")),
        ("Depth",     ("Depth", "S[table-format=1.2]", "S[table-format=1.2]", "S[table-format=1.2]")),
        ("Halstead_V",("V", "S[table-format=2.2]", "S[table-format=2.2]", "S[table-format=2.2]")),
        ("Halstead_D",("D", "S[table-format=1.2]", "S[table-format=1.2]", "S[table-format=1.2]")),
        ("Halstead_E",("E", "S[table-format=3.2]", "S[table-format=3.2]", "S[table-format=3.2]")),
    ]

    # Right bars for each numeric column (15 cols total): TS(3), Depth(3), V(3), D(3), E(3)
    # Matches the table spec: groups end with "||" except the last group (E) which ends with "|"
    right_bars = [
        "|", "|", "||",   # TS: avg, min, max
        "|", "|", "||",   # Depth
        "|", "|", "||",   # V
        "|", "|", "||",   # D
        "|", "|", "|"     # E
    ]

    def latex_escape(s: str) -> str:
        if s is None:
            return ""
        s = str(s)
        return (s.replace("\\", r"\textbackslash{}")
                 .replace("&", r"\&")
                 .replace("%", r"\%")
                 .replace("$", r"\$")
                 .replace("#", r"\#")
                 .replace("_", r"\_")
                 .replace("{", r"\{")
                 .replace("}", r"\}")
                 .replace("~", r"\textasciitilde{}")
                 .replace("^", r"\textasciicircum{}"))

    def stat_triplet(series):
        series = series.dropna()
        if len(series) == 0:
            return (None, None, None)
        return (series.mean(), series.min(), series.max())

    def fmt_num(x):
        return f"{float(x):.2f}"

    def cell(x, rb):
        # rb is "|" or "||"
        if x is None:
            return rf"\multicolumn{{1}}{{c{rb}}}{{---}}"
        return fmt_num(x)

    def hhline_between_rows(ncols: int) -> str:
        # First column is multirow, so use "~" there, then "-" for the rest
        # Example target: \hhline{|~|-|-|-|...|}
        parts = ["~"] + ["-"] * (ncols - 1)
        return r"\hhline{|" + "|".join(parts) + "|}"

    lines = []
    lines.append(r"% Requires (at least): \usepackage{siunitx,multirow,hhline,colortbl,xcolor}")
    lines.append(r"\begin{table}[h!]")
    lines.append(r"  \centering")
    lines.append(r"  \caption{\revised{Complexity summary by domain and source. TS: Tree Size. $V$, $D$, $E$: Halstead Volume, Difficulty, and Effort.}}")
    lines.append(r"  \resizebox{\textwidth}{!}{%")

    # Column spec exactly like your target (with line breaks for readability)
    colspec = (
        r"    \begin{tabular}{|l|l|"
        r"      S[table-format=2.2]|S[table-format=2.2]|S[table-format=2.2]||"
        r"      S[table-format=1.2]|S[table-format=1.2]|S[table-format=1.2]||"
        r"      S[table-format=2.2]|S[table-format=2.2]|S[table-format=2.2]||"
        r"      S[table-format=1.2]|S[table-format=1.2]|S[table-format=1.2]||"
        r"      S[table-format=3.2]|S[table-format=3.2]|S[table-format=3.2]|}"
    )
    lines.append(colspec)

    # Header row 1
    lines.append(r"      \hline")
    lines.append(r"      \rowcolor{gray!15}")
    lines.append(r"      \textbf{Case} & \textbf{Source} &")
    lines.append(r"      \multicolumn{3}{c||}{\textbf{TS}} &")
    lines.append(r"      \multicolumn{3}{c||}{\textbf{Depth}} &")
    lines.append(r"      \multicolumn{3}{c||}{\textbf{$V$}} &")
    lines.append(r"      \multicolumn{3}{c||}{\textbf{$D$}} &")
    lines.append(r"      \multicolumn{3}{c|}{\textbf{$E$}} \\")
    lines.append(r"      \hline")

    # Header row 2
    lines.append(r"      \rowcolor{gray!30}")
    lines.append(r"      & &")
    lines.append(r"      \multicolumn{1}{c|}{avg} & \multicolumn{1}{c|}{min} & \multicolumn{1}{c||}{max} &")
    lines.append(r"      \multicolumn{1}{c|}{avg} & \multicolumn{1}{c|}{min} & \multicolumn{1}{c||}{max} &")
    lines.append(r"      \multicolumn{1}{c|}{avg} & \multicolumn{1}{c|}{min} & \multicolumn{1}{c||}{max} &")
    lines.append(r"      \multicolumn{1}{c|}{avg} & \multicolumn{1}{c|}{min} & \multicolumn{1}{c||}{max} &")
    lines.append(r"      \multicolumn{1}{c|}{avg} & \multicolumn{1}{c|}{min} & \multicolumn{1}{c|}{max} \\")
    lines.append(r"      \hline")
    lines.append("")

    # Total columns = 17 (Case, Source, 15 numeric)
    ncols_total = 17
    midrule = hhline_between_rows(ncols_total)

    for di, domain in enumerate(domains):
        sub = df[df["Domain"] == domain]
        human = sub[sub["Origin"] == "Human"]
        ai = sub[sub["Origin"] == "AI"]

        n_h = len(human)
        n_a = len(ai)

        # Build numeric cells: TS(avg,min,max), Depth(...), V(...), D(...), E(...)
        def build_numeric_cells(block):
            cells = []
            for (col, _) in metric_map:
                avg, mn, mx = stat_triplet(block[col]) if col in block.columns else (None, None, None)
                cells.extend([avg, mn, mx])
            # Format with correct right-bar handling for missing values
            out = []
            for x, rb in zip(cells, right_bars):
                out.append(cell(x, rb))
            return " & ".join(out)

        h_nums = build_numeric_cells(human)
        a_nums = build_numeric_cells(ai)

        dom = latex_escape(domain)

        # Domain block
        lines.append(rf"      \multirow{{2}}{{*}}{{{dom}}}")
        lines.append(rf"      & Human (n={n_h}) & {h_nums} \\")
        lines.append(rf"      {midrule}")
        lines.append(rf"      & AI (n={n_a})   & {a_nums} \\")
        lines.append(r"      \hline")
        lines.append("")

        # Spacing between domains (except last), matching your target style
        if di < len(domains) - 1:
            lines.append(r"      \noalign{\vskip\doublerulesep}")
            lines.append(r"      \hline")
            lines.append("")

    lines.append(r"    \end{tabular}%")
    lines.append(r"  }")
    lines.append(r"  \label{tab:complexity_summary_by_domain_source}")
    lines.append(r"\end{table}")

    with open(output_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines) + "\n")

    print(f"  Saved: {output_path}")

In [182]:
# ============================================================
# MAIN PROCESSING

In [183]:
# ============================================================
def process_domain(filepath):
    """Process one domain file. Returns metrics DataFrame."""
    domain = Path(filepath).stem.replace('mrs_', '')
    print(f'\n{"=" * 60}')
    print(f'Processing: {filepath} ({domain})')
    print(f'{"=" * 60}')

    trees, raw_texts, origins, errors = load_mrs_from_file(filepath)
    print(f'  MRs loaded: {len(trees)}')
    n_human = sum(1 for v in origins.values() if v == 'Human')
    n_ai = len(trees) - n_human
    print(f'  Human: {n_human}, AI: {n_ai}')
    if errors:
        print(f'  Parse errors: {len(errors)}')
        for fid, raw, err in errors:
            print(f'    {fid}: {err}')

    if not trees:
        return None, domain

    # Show examples
    print('\n  --- Tree Examples ---')
    for mr_id, tree in list(trees.items())[:2]:
        print(f'\n  {mr_id}:')
        tree.print_tree()

    # Compute metrics
    rows = []
    for mr_id, tree in trees.items():
        raw = raw_texts.get(mr_id, '')
        row = compute_all_metrics(mr_id, tree, raw)
        row['Origin'] = origins.get(mr_id, 'AI')
        rows.append(row)

    df = pd.DataFrame(rows)
    df['Domain'] = domain

    # Print summary
    print(f'\n  --- Complexity Summary ---')
    print(f'  Halstead V: mean={df["Halstead_V"].mean():.2f}, '
          f'std={df["Halstead_V"].std():.2f}, '
          f'range=[{df["Halstead_V"].min():.2f}, {df["Halstead_V"].max():.2f}]')
    print(f'  McCabe CC:  mean={df["McCabe_CC"].mean():.2f}, '
          f'range=[{df["McCabe_CC"].min()}, {df["McCabe_CC"].max()}]')
    print(f'  Vocabulary: mean={df["n_vocabulary"].mean():.2f}, '
          f'range=[{df["n_vocabulary"].min()}, {df["n_vocabulary"].max()}]')

    # Save per-domain outputs
    domain_dir = os.path.join(OUTPUT_DIR, domain)
    os.makedirs(domain_dir, exist_ok=True)

    csv_path = os.path.join(domain_dir, f'{domain}_complexity.csv')
    df.to_csv(csv_path, index=False)
    print(f'  Saved: {csv_path}')

    report_path = os.path.join(domain_dir, f'{domain}_complexity_report.txt')
    generate_report(domain, trees, raw_texts, df, report_path)

    return df, domain

In [184]:
# ============================================================
# EXECUTION

In [185]:
# ============================================================

In [186]:
if __name__ == '__main__':
    print('=' * 60)
    print('MR COMPLEXITY ANALYSIS (UNSIMPLIFIED TREES)')
    print('=' * 60)

    all_dfs = []
    for fp in INPUT_FILES:
        if os.path.exists(fp):
            df, domain = process_domain(fp)
            if df is not None:
                all_dfs.append(df)
        else:
            print(f'\n  File not found: {fp}')

    if all_dfs:
        combined = pd.concat(all_dfs, ignore_index=True)

        # Reorder columns: Domain first, Origin, MR_ID, ..., Raw_MR last
        cols = list(combined.columns)
        for c in ['Domain', 'Origin', 'MR_ID', 'Raw_MR']:
            if c in cols:
                cols.remove(c)
        ordered_cols = ['Domain', 'Origin', 'MR_ID'] + cols
        if 'Raw_MR' in combined.columns:
            ordered_cols.append('Raw_MR')
        combined = combined[ordered_cols]

        # Write CSV with legend at the bottom
        csv_path = os.path.join(OUTPUT_DIR, 'all_domains_complexity.csv')
        combined.to_csv(csv_path, index=False)

        # Append legend
        legend = [
            '',
            'LEGEND',
            'Domain,SUT domain (AVs / DCs / DFAs / Words)',
            'Origin,Human-written or AI-generated MR',
            'MR_ID,Metamorphic Relation identifier (group-category-MRn)',
            'Tree_Size,Total number of nodes in the unsimplified parse tree',
            'Depth,Maximum depth of the parse tree',
            'Leaves,Number of leaf nodes (operands)',
            'Branching_Factor,Average branching factor = edges / internal nodes',
            'McCabe_CC,"Cyclomatic Complexity = #OR + 1 (McCabe, 1976)"',
            'N1_operators,Total operator tokens (AND / OR / COMP / PRED / NEG_PRED)',
            'N2_operands,"Total operand tokens (original expressions: NS(m1), TTD(m2), etc.)"',
            'n1_unique_ops,Number of distinct operator types',
            'n2_unique_opds,Number of distinct operand expressions',
            'N_length,Halstead program length = N1 + N2',
            'n_vocabulary,Halstead vocabulary = n1 + n2',
            'Halstead_V,"Volume = N * log2(n) in bits (Halstead, 1977)"',
            'Halstead_D,"Difficulty = (n1/2) * (N2/n2) (Halstead, 1977)"',
            'Halstead_E,"Effort = D * V (Halstead, 1977)"',
            'Operators_detail,List of operator tokens found in the tree',
            'Operands_detail,List of operand tokens found in the tree',
            'Raw_MR,Original MR text as written in the input file',
        ]
        with open(csv_path, 'a', encoding='utf-8') as f:
            f.write('\n'.join(legend) + '\n')

        print(f'\n  Saved: {csv_path}')

        # Summary table PDF
        pdf_path = os.path.join(OUTPUT_DIR, 'complexity_table.pdf')
        generate_table_pdf(combined, pdf_path)

        # LaTeX summary table
        latex_path = os.path.join(OUTPUT_DIR, 'complexity_summary_table.tex')
        generate_latex_table(combined, latex_path)

    print('\n' + '=' * 60)
    print('SUMMARY')
    print('=' * 60)
    for df_domain in all_dfs:
        domain = df_domain['Domain'].iloc[0]
        print(f'\n  {domain}: {len(df_domain)} MRs')
        print(f'    Halstead V: {df_domain["Halstead_V"].mean():.2f} '
              f'(std {df_domain["Halstead_V"].std():.2f})')
        print(f'    McCabe CC:  {df_domain["McCabe_CC"].mean():.2f}')
        print(f'    Vocabulary: {df_domain["n_vocabulary"].mean():.1f}')
        print(f'    Tree Size:  {df_domain["Tree_Size"].mean():.1f}')

MR COMPLEXITY ANALYSIS (UNSIMPLIFIED TREES)

Processing: mrs_DCs_human_extra.txt (DCs_human_extra)
  MRs loaded: 13
  Human: 5, AI: 8

  --- Tree Examples ---

  example-human-MR1:
└── MR
    ├── LHS
    │   └── COMP(>)
    │       ├── NN(m1)
    │       └── NN(m2)
    └── RHS
        └── COMP(<=)
            ├── T(m1)
            └── T(m2)

  example-human-MR2:
└── MR
    ├── LHS
    │   └── AND
    │       ├── COMP(>)
    │       │   ├── CPU(m1)
    │       │   └── CPU(m2)
    │       └── COMP(==)
    │           ├── W(m1)
    │           └── W(m2)
    └── RHS
        └── COMP(<=)
            ├── E(m1)
            └── E(m2)

  --- Complexity Summary ---
  Halstead V: mean=19.48, std=7.85, range=[13.93, 33.22]
  McCabe CC:  mean=1.00, range=[1, 1]
  Vocabulary: mean=6.85, range=[5, 10]
  Saved: output_complexity/DCs_human_extra/DCs_human_extra_complexity.csv
  Saved: output_complexity/DCs_human_extra/DCs_human_extra_complexity_report.txt

Processing: mrs_AVs_human_extra.txt (AVs_human